# Capstone — Derive the null before reading the grid

A worked example from the repo's own record (P-1, resolved as R-1 in
`PREDICTIONS.md`; `notes/p1_decomposition.md` D4 hole (a);
`notes/p4_twisted_inertial_ring.md` run 5).

1. **d4_nopin**: a Kuramoto lattice with no per-site drive, twisted and
   control, 101 values of $\Omega$. The first reading counted 7 "rational
   snaps" per row. The null: $\rho = \Omega$ identically (claim
   `klein-twisted-mean-frequency-identity`; each undirected edge contributes
   $\sin x + \sin(-x) = 0$), so a "snap" is any grid point where $\Omega$ itself
   is a rational with $q \le 8$. Computed below: the snap set equals that
   grid set.
2. **P-4 run 5**: slip periods near 3.0 (and 4.3) round trips were read as
   integer plateaus. The null: quasi-static loading with stiffness $4J/N$
   gives period/round-trip $= F_N/(4Jv)$ with no wave involved (catalog c12,
   within 25% on the 16 rows with prediction $\ge 3$); $F_N/(4v) = 3.0$ at
   $(0.6, 0.05)$ and $(1.2, 0.1)$ - a grid coincidence, flagged in the table.
   Catalog c13 records what the formula cannot see (a $2.2\times$ swing with
   $\mu_d$ at fixed $F_N, v$).

In [ ]:
import sys, math, json, cmath, random
from fractions import Fraction
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CATALOG.md").exists())
sys.path.insert(0, str(_root / "notebooks"))
from nbkit import ROOT, show_svg, catalog, verify, mutant_must_fail, falsify
import termplot
print("repo root:", ROOT.name)

## 1. The snaps are the grid

In [ ]:
rows = json.loads((ROOT / "scripts/experiments/d4_nopin_results.json").read_text())
TOL = 5e-4
def snaps(rows):
    out = []
    for r in rows:
        for q in range(2, 9):
            p = round(r["rho"] * q)
            if abs(r["rho"] - p / q) < TOL and math.gcd(p, q) == 1 and 0 < p < q:
                out.append(round(r["Omega"], 2)); break
    return sorted(set(out))

grid_rationals = sorted({k / 100 for k in range(101)
                         if 0 < k < 100 and Fraction(k, 100).denominator <= 8})
print("grid points that ARE rationals with q <= 8 (the null):", grid_rationals)
maxdev = 0.0
for Delta in (0.05, 0.2):
    for tw in (False, True):
        sel = [r for r in rows if r["Delta"] == Delta and r["twisted"] == tw]
        s = snaps(sel)
        maxdev = max(maxdev, max(abs(r["rho"] - r["Omega"]) for r in sel))
        print(f"Delta {Delta:<5} {'twisted' if tw else 'control':>8}: snaps {s} == null: {s == grid_rationals}")
print(f"max |rho - Omega| over all {len(rows)} runs: {maxdev:.1e}")

In [ ]:
def check(tol=TOL, null="grid"):
    ok = abs(maxdev) < 1e-10
    for Delta in (0.05, 0.2):
        for tw in (False, True):
            sel = [r for r in rows if r["Delta"] == Delta and r["twisted"] == tw]
            s = snaps(sel)
            ok &= (s == grid_rationals) if null == "grid" else (len(s) > 2.2 and s != grid_rationals)
    return ok

falsify(check, {"snaps-are-locking": lambda: {"null": "locking"}})

The identity behind the null, with its own named mutant:

In [ ]:
rc, out = verify("p1_mean_frequency.py")
assert rc == 0
rc, out = verify("p1_mean_frequency.py", mutant="phase-lag")
mutant_must_fail("phase-lag", rc, out)

## 2. The loading formula

In [ ]:
data = json.loads((ROOT / "scripts/experiments/p4_results_lowdamp.json").read_text())
sel = [r for r in data if r["g"] == 0.01 and r["regime"] == "stick-slip" and r["F_N"] / (4 * r["v"]) >= 3 - 1e-9]
print(f"{'N':>3} {'F_N':>5} {'v':>5} | observed  F_N/(4Jv)  rel.err")
errs = []
for r in sorted(sel, key=lambda r: (r["N"], r["v"], r["F_N"])):
    pred = r["F_N"] / (4 * r["v"])
    e = abs(r["ratio"] - pred) / r["ratio"]
    errs.append(e)
    flag = "  <- F_N/(4v) = 3.0 on the grid" if abs(pred - 3.0) < 1e-9 else ("  <- read as '4.3'" if abs(r["ratio"] - 4.3) < 0.1 else "")
    print(f"{r['N']:>3} {r['F_N']:>5} {r['v']:>5} | {r['ratio']:8.3f}  {pred:8.3f}  {e:6.2f}{flag}")
print(f"{len(sel)} rows; worst relative error {max(errs):.2f}")
print(termplot.plot_xy([(r['F_N'] / (4 * r['v']), r['ratio']) for r in sel] + [(x / 2, x / 2) for x in range(4, 30)],
                       width=50, height=12, title="observed period vs F_N/(4Jv) (diagonal = formula)", xlabel="F/(4Jv)", ylabel="period"))

In [ ]:
mud = json.loads((ROOT / "scripts/experiments/p4_mud_results.json").read_text())["rows"]
print("mu_d dependence at fixed (N, g, v, F_N) where the formula predicts 4.5 for every row:")
for r in mud:
    print(f"   mu_d = {r['mu_d']}: period {r['ratio']}")

## Falsifiers: c12 and c13 mutants must fail

In [ ]:
for entry in ("c12_p4_loading_formula", "c13_p4_mud_dependence"):
    rc, _ = catalog(entry)
    assert rc == 0, entry
    rc, out = catalog(entry, mutant=True)
    mutant_must_fail(entry, rc, out)

The lesson both rows share (P-4 note, run 5): a plateau read off a grid is
not a plateau until the null that would put it there has been derived and
ruled out. That is the same discipline as LAW-11: a check without a failing
mutant is a restatement.